# Generate Enhanced Queries: Gemma 3 4B-IT

**Model:** google/gemma-3-4b-it

**Hardware:** A100 GPU (40GB VRAM)

**Quantization:** None (FP16)

**Temperature:** 0.1 (more focused)

**Batch Size:** 16 (optimized for A100)

**Note:** This is a GATED model - you must accept the license and login to HuggingFace

---

## Setup

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21 (required for Pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch
!pip install -q datasets accelerate bitsandbytes huggingface_hub

print("\n" + "="*60)
print("Installation complete")
print("="*60)
print("IMPORTANT: Restart runtime now!")
print("   1. Click 'Runtime' -> 'Restart runtime'")
print("   2. Then run cells starting from 'Step 2' below")
print("="*60)

Cloning into 'graduation'...
remote: Enumerating objects: 552, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 552 (delta 31), reused 73 (delta 23), pack-reused 468 (from 1)
Receiving objects: 100% (552/552), 20.54 MiB | 9.13 MiB/s, done.
Resolving deltas: 100% (203/203), done.
/content/graduation/arabic-rag-query-enhancement
Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /usr/lib/j

### Step 2: Mount Google Drive and Configure Environment (Run After Restart)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project
%cd /content/graduation/arabic-rag-query-enhancement

# Configure environment
import os
import sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Add src to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

print("\nEnvironment configured")
print("Ready to run experiment")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/graduation/arabic-rag-query-enhancement
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)

Environment configured
Ready to run experiment


### Step 3: HuggingFace Login (REQUIRED for Gemma 3)

Gemma 3 is a gated model. You must:
1. Accept the license at: https://huggingface.co/google/gemma-3-4b-it
2. Get your HuggingFace token from: https://huggingface.co/settings/tokens
3. Run the cell below and paste your token when prompted

In [ ]:
from huggingface_hub import login

# Login to HuggingFace (you'll be prompted for token)
login()

print("\nLogged in to HuggingFace")
print("\nIMPORTANT: Make sure you've accepted the Gemma license at:")
print("https://huggingface.co/google/gemma-3-4b-it")
print("\nIf you haven't accepted it, the model download will fail.")


Logged in to HuggingFace

IMPORTANT: Make sure you've accepted the Gemma license at:
https://huggingface.co/google/gemma-3-4b-it

If you haven't accepted it, the model download will fail.


## Import Modules

In [ ]:
from src.utils.data_loader import MIRACLDataLoader
from src.enhancers.query2doc import Query2DocEnhancer

import torch
from tqdm.notebook import tqdm

print("Modules imported")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Modules imported
GPU Available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 39.5 GB


## Load Data

In [ ]:
# Load MIRACL Arabic dev set
data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

print(f"\nDataset Statistics:")
print(f"  Queries: {len(topics)}")
print(f"  Qrels: {len(qrels)}")

# Show sample
sample_qid = list(topics.keys())[0]
print(f"\nSample Query:")
print(f"  ID: {sample_qid}")
print(f"  Text: {topics[sample_qid]['title']}")
print(f"  Relevant docs: {len(qrels.get(sample_qid, {}))}")

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries

Dataset Statistics:
  Queries: 2896
  Qrels: 2896

Sample Query:
  ID: 8099
  Text: من هو علي بن محمد السمري؟
  Relevant docs: 10


## Initialize Gemma 3 4B Enhancer

In [ ]:
# 1. Clear GPU memory completely
import torch
import gc

del enhancer
torch.cuda.empty_cache()
gc.collect()

print("✓ GPU memory cleared")


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# After restart and environment setup, run this instead:

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading Gemma 3 4B with bfloat16 (more stable)...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# Load model with bfloat16 (more numerically stable than float16)
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-4b-it",
    torch_dtype=torch.bfloat16,  # Changed from float16
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"  # Avoid flash attention issues
)
model.eval()

print(f"✓ Model loaded on {model.device}")
print(f"✓ Using bfloat16 for numerical stability")


Loading Gemma 3 4B with bfloat16 (more stable)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

✓ Model loaded on cuda:0
✓ Using bfloat16 for numerical stability


## Check GPU Memory

In [ ]:
if torch.cuda.is_available():
    print("=== GPU Status ===")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"Memory total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3:.2f} GB")

=== GPU Status ===
GPU: NVIDIA A100-SXM4-40GB
Memory allocated: 8.01 GB
Memory reserved: 8.03 GB
Memory total: 39.49 GB
Memory free: 31.48 GB


In [ ]:
# Create enhancer with stability fixes
class SimpleGemma3Enhancer:
    def __init__(self, model, tokenizer, max_new_tokens=128, temperature=0.1, batch_size=16):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = max(temperature, 0.01)  # Prevent temperature=0
        self.top_p = 0.9
        self.batch_size = batch_size
        self.system_prompt = (
            "You are asked to write a passage that answers the given query. "
            "Do not ask the user for further clarification. "
            "Respond in Arabic only."
        )

    def enhance(self, query, query_id=None):
        """Enhance single query"""
        prompt = f"{self.system_prompt}\n\nQuery: {query}\n\nAnswer:"

        inputs = self.tokenizer(prompt, return_tensors="pt", padding=True).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                top_p=self.top_p,
                do_sample=True if self.temperature > 0.01 else False,  # Greedy if temp too low
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                repetition_penalty=1.1,  # Prevent repetition
                no_repeat_ngram_size=3   # Prevent exact repeats
            )

        generated = outputs[0][inputs.input_ids.shape[1]:]
        pseudo_doc = self.tokenizer.decode(generated, skip_special_tokens=True)

        return f"{query} {pseudo_doc}"

    def enhance_batch_parallel(self, queries, query_ids=None):
        """Enhance batch of queries"""
        prompts = [f"{self.system_prompt}\n\nQuery: {q}\n\nAnswer:" for q in queries]

        inputs = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                top_p=self.top_p,
                do_sample=True if self.temperature > 0.01 else False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3
            )

        input_length = inputs.input_ids.shape[1]
        pseudo_docs = self.tokenizer.batch_decode(outputs[:, input_length:], skip_special_tokens=True)

        return [f"{q} {doc}" for q, doc in zip(queries, pseudo_docs)]

    def enhance_batch(self, queries, query_ids=None, show_progress=True):
        """Enhance with batching"""
        from tqdm.notebook import tqdm

        enhanced = []
        num_batches = (len(queries) + self.batch_size - 1) // self.batch_size

        iterator = tqdm(range(num_batches), desc="Enhancing batches") if show_progress else range(num_batches)

        for batch_idx in iterator:
            start_idx = batch_idx * self.batch_size
            end_idx = min(start_idx + self.batch_size, len(queries))
            batch_queries = queries[start_idx:end_idx]
            batch_enhanced = self.enhance_batch_parallel(batch_queries)
            enhanced.extend(batch_enhanced)

        return enhanced

# Create enhancer with temperature 0.1
enhancer = SimpleGemma3Enhancer(model, tokenizer, max_new_tokens=128, temperature=0.1, batch_size=16)

print("✓ Gemma 3 enhancer created with stability fixes")
print("✓ Using bfloat16, repetition penalty, and safe temperature")


✓ Gemma 3 enhancer created with stability fixes
✓ Using bfloat16, repetition penalty, and safe temperature


## Test on Sample Query

In [ ]:
# Test enhancer on sample query
sample_query = topics[sample_qid]['title']
print(f"Testing enhancer on sample query...\n")
print(f"Original: {sample_query}")
print(f"\nGenerating pseudo-document...")

enhanced_sample = enhancer.enhance(sample_query)
print(f"\nEnhanced: {enhanced_sample[:500]}...")  # Show first 500 chars
print(f"\nLength: {len(sample_query)} -> {len(enhanced_sample)} chars")
print(f"Expansion ratio: {len(enhanced_sample)/len(sample_query):.2f}x")

Testing enhancer on sample query...

Original: من هو علي بن محمد السمري؟

Generating pseudo-document...

Enhanced: من هو علي بن محمد السمري؟ 

علي بن محمد الأسلمي، أبو بكر، عالم دين ومحدث وصحيح حديث من أهل بغداد. ولد في بغداد عام 670 هـ وتوفي فيها عام 750 ه‍. اشتهر بعلمه وفقهه وصدقه، وكان من أئمة البلاط العباسيين. له مؤلفات عديدة منها "التحذير من الخطأ" و "مجمع البيان".

‏علي بن محمّد الأسلميّ، أَبُو بكرٍ، عالمُ ديّنٍ ومُحدِّثٌ وصحيَّ...

Length: 25 -> 323 chars
Expansion ratio: 12.92x


## Generate Enhanced Queries for All Data

In [ ]:
import time

print("="*60)
print("GENERATING ENHANCED QUERIES: Gemma 3 4B-IT")
print("="*60)

# Prepare queries
query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nTotal queries: {len(query_texts)}")
print(f"Batch size: 16")
print(f"Expected batches: {len(query_texts) // 16 + 1}")
print(f"Expected time: ~20-25 minutes\n")

start_time = time.time()

# Apply Query2Doc enhancement
enhanced_queries = enhancer.enhance_batch(
    query_texts,
    query_ids,
    show_progress=True
)

elapsed = time.time() - start_time

print(f"\nEnhanced {len(enhanced_queries)} queries")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Queries per minute: {len(query_texts)/(elapsed/60):.1f}")

GENERATING ENHANCED QUERIES: Gemma 3 4B-IT

Total queries: 2896
Batch size: 16
Expected batches: 182
Expected time: ~20-25 minutes



Enhancing batches:   0%|          | 0/181 [00:00<?, ?it/s]


Enhanced 2896 queries
Total time: 33.8 minutes
Queries per minute: 85.7


## Show Enhancement Examples

In [ ]:
print("\nEnhancement Examples:\n")
for i in range(min(5, len(query_texts))):
    print(f"Query {i+1}:")
    print(f"  Original ({len(query_texts[i])} chars): {query_texts[i]}")
    print(f"  Enhanced ({len(enhanced_queries[i])} chars): {enhanced_queries[i][:200]}...")  # First 200 chars
    print(f"  Expansion: {len(enhanced_queries[i])/len(query_texts[i]):.2f}x")
    print()


Enhancement Examples:

Query 1:
  Original (25 chars): من هو علي بن محمد السمري؟
  Enhanced (420 chars): من هو علي بن محمد السمري؟ 

علي بن محمد الأسلمي، المعروف بالسمري، هو أحد أشهر وأهم علماء اللغة العربية في العصر الذهبي للإسلام. ولد في مكة المكرمة حوالي عام 150 هـ، وتوفي فيها أيضًا. اشتهر بكونه اللغو...
  Expansion: 16.80x

Query 2:
  Original (34 chars): متى تم إستخدام الغوّاصات لأول مرة؟
  Enhanced (438 chars): متى تم إستخدام الغوّاصات لأول مرة؟ 
تم استخدام الغواصات لأغراض عسكرية لأول مـرة في أواخر القرن الثامن عشر، حيث قام الكابتن جون دافنبورت بإنشاء غواصة صغيرة تعمل بالهواء المضغوط في بريطانيا العظمى عام 1...
  Expansion: 12.88x

Query 3:
  Original (28 chars): من هو القديس المسمى بالصخرة؟
  Enhanced (374 chars): من هو القديس المسمى بالصخرة؟ 
القديس الذي يُعرف بالصّخرة هو القِدِّيسُ سمعان الخَطّاب (Saint Simeon the Stylite). كان راهبًا يونانيًا في القرن الثالث الميلادي، اشتهر بتقوّمه بالصوم والتَّفريط والعبادة...
  Expansion: 13.36x

Query 4:
  Original (33 chars): هل يرتبط ال

## Query Expansion Statistics

In [ ]:
import numpy as np

# Calculate statistics
original_lengths = [len(q) for q in query_texts]
enhanced_lengths = [len(eq) for eq in enhanced_queries]
expansion_ratios = [e/o if o > 0 else 0 for o, e in zip(original_lengths, enhanced_lengths)]

print("=== Query Expansion Statistics ===")
print(f"\nOriginal queries:")
print(f"  Mean length: {np.mean(original_lengths):.1f} chars")
print(f"  Median length: {np.median(original_lengths):.1f} chars")
print(f"  Min/Max: {min(original_lengths)} / {max(original_lengths)} chars")

print(f"\nEnhanced queries:")
print(f"  Mean length: {np.mean(enhanced_lengths):.1f} chars")
print(f"  Median length: {np.median(enhanced_lengths):.1f} chars")
print(f"  Min/Max: {min(enhanced_lengths)} / {max(enhanced_lengths)} chars")

print(f"\nExpansion ratio:")
print(f"  Mean: {np.mean(expansion_ratios):.2f}x")
print(f"  Median: {np.median(expansion_ratios):.2f}x")
print(f"  Min/Max: {min(expansion_ratios):.2f}x / {max(expansion_ratios):.2f}x")

=== Query Expansion Statistics ===

Original queries:
  Mean length: 29.5 chars
  Median length: 27.0 chars
  Min/Max: 12 / 101 chars

Enhanced queries:
  Mean length: 296.1 chars
  Median length: 351.0 chars
  Min/Max: 42 / 624 chars

Expansion ratio:
  Mean: 11.38x
  Median: 10.47x
  Min/Max: 1.24x / 37.29x


## Save Enhanced Queries

In [ ]:
import pickle

# Save enhanced queries
output_file = 'enhanced_queries_gemma3_4b.pkl'

with open(output_file, 'wb') as f:
    pickle.dump({
        'query_ids': query_ids,
        'original': query_texts,
        'enhanced': enhanced_queries,
        'model': 'google/gemma-3-4b-it',
        'config': {
            'max_new_tokens': 128,
            'temperature': 0.1,
            'top_p': 0.9,
            'batch_size': 16,
            'quantization': 'None (FP16)',
            'hardware': 'A100 GPU'
        },
        'stats': {
            'total_queries': len(query_texts),
            'mean_original_length': np.mean(original_lengths),
            'mean_enhanced_length': np.mean(enhanced_lengths),
            'mean_expansion_ratio': np.mean(expansion_ratios),
            'generation_time_minutes': elapsed/60
        }
    }, f)

print(f"Enhanced queries saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024**2:.1f} MB")

# Also save to Google Drive
drive_path = '/content/drive/MyDrive/enhanced_queries_gemma3_4b.pkl'
!cp {output_file} {drive_path}
print(f"\nBackup saved to Google Drive: {drive_path}")

Enhanced queries saved to: enhanced_queries_gemma3_4b.pkl
File size: 1.6 MB

Backup saved to Google Drive: /content/drive/MyDrive/enhanced_queries_gemma3_4b.pkl


## Summary

In [ ]:
print("="*60)
print("GENERATION COMPLETE")
print("="*60)
print(f"\nModel: Gemma 3 4B-IT")
print(f"Quantization: None (FP16)")
print(f"Temperature: 0.1")
print(f"Batch size: 16")
print(f"Hardware: {torch.cuda.get_device_name(0)}")
print(f"\nQueries processed: {len(enhanced_queries)}")
print(f"Generation time: {elapsed/60:.1f} minutes")
print(f"Average expansion: {np.mean(expansion_ratios):.2f}x")
print(f"\nOutput file: {output_file}")
print(f"\nNote: Gemma 3 has weaker Arabic performance than other models.")
print(f"This serves as a lower-bound comparison.")
print(f"\nNext step: Use evaluate_enhanced_queries.ipynb to test with Dense and BM25")